In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [13]:
# Define State
class State(TypedDict):
    query : str
    tool_result : str
    answer : str

In [14]:
# Define tool
def calculator_tool():
    return "2+2=4"

# Define Router
def route_query(state: State) -> str:
    if "add" in state["query"].lower():
        return "calculator"
    else:
        return "answer"

In [18]:
# Create Node

def calculator_node(state: State) -> State:
    result = calcultor_tool()
    
    return {
        "tool_result": result
    }
    
def answer_node(state: State) -> State:
    
    if state.get("tool_result") is not None:
        return {
            "answer": state["tool_result"]
        }
    return {
        "answer": "Answered directly by LLM"
    }

In [19]:
# Build Graph

builder = StateGraph(State)

builder.add_node(
    "router",
    lambda state:{}
)

builder.add_node(
    "calculator",
    calculator_node
)

builder.add_node(
    "answer",
    answer_node
)

builder.add_edge(START, "router")

builder.add_conditional_edges("router", route_query)

builder.add_edge("calculator", "answer")

builder.add_edge("answer", END)

graph = builder.compile()

In [22]:
graph.invoke({
    "query": "add"})

{'query': 'add', 'tool_result': '2+2=4', 'answer': '2+2=4'}